# Zepto Inventory Analysis — Phase 1: Data Wrangling

**Dataset:** Zepto product listings scraped from public app data  
**Rows:** 3,732 | **Columns:** 9  
**Goal:** Clean raw data, fix data types, engineer new columns, export analysis-ready CSV


In [1]:
import pandas as pd
import numpy as np
import os

# Display settings
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.2f}'.format)

print("Libraries loaded.")

Libraries loaded.


In [10]:
# Load data
df = pd.read_excel('../data/zepto_v1.xlsx')
print(f"Shape: {df.shape}")
print(f"Rows: {df.shape[0]} | Columns: {df.shape[1]}")


Shape: (3732, 9)
Rows: 3732 | Columns: 9


In [3]:
# See raw data
df.head()

,Category,name,mrp,discountPercent,availableQuantity,discountedSellingPrice,weightInGms,outOfStock,quantity
0,Fruits & Vegetables,Onion,2500,16,3,2100,1000,False,1
1,Fruits & Vegetables,Tomato Hybrid,4200,16,3,3500,1000,False,1
2,Fruits & Vegetables,Tender Coconut,5100,15,3,4300,58,False,1
3,Fruits & Vegetables,Coriander Leaves,2000,15,3,1700,100,False,100
4,Fruits & Vegetables,Ladies Finger,1400,14,3,1200,250,False,250


In [4]:
print("=== COLUMN TYPES ===")
print(df.dtypes)
print("\n=== NULL COUNT ===")
print(df.isnull().sum())
print("\n=== BASIC STATS ===")
df.describe()

=== COLUMN TYPES ===
Category                  object
name                      object
mrp                        int64
discountPercent            int64
availableQuantity          int64
discountedSellingPrice     int64
weightInGms                int64
outOfStock                  bool
quantity                   int64
dtype: object

=== NULL COUNT ===
Category                  0
name                      0
mrp                       0
discountPercent           0
availableQuantity         0
discountedSellingPrice    0
weightInGms               0
outOfStock                0
quantity                  0
dtype: int64

=== BASIC STATS ===


,mrp,discountPercent,availableQuantity,discountedSellingPrice,weightInGms,quantity
count,3732.00,3732.00,3732.00,3732.00,3732.00,3732.00
mean,15680.12,7.62,4.01,14192.83,387.84,213.27
std,16088.81,9.21,2.20,13850.73,678.10,194.73
min,0.00,0.00,0.00,0.00,0.00,0.00
25%,6000.00,0.00,2.00,5500.00,100.00,50.00
50%,11000.00,6.00,5.00,10400.00,225.00,186.00
75%,20000.00,10.00,6.00,18400.00,450.00,340.00
max,260000.00,51.00,6.00,139900.00,10000.00,1500.00


## Step 1: Fix Data Types

Raw data issues found:

- `mrp` and `discountedSellingPrice` stored in **paise** (divide by 100)
- `discountPercent` is int — fine as-is


In [11]:
#  Convert prices paise → rupees
# Paise to rupees (divide by 100)
df['mrp'] = df['mrp'] / 100
df['discountedSellingPrice'] = df['discountedSellingPrice'] / 100

print("Price columns converted to ₹")
print(f"MRP range: ₹{df['mrp'].min():.2f} to ₹{df['mrp'].max():.2f}")
print(f"Selling price range: ₹{df['discountedSellingPrice'].min():.2f} to ₹{df['discountedSellingPrice'].max():.2f}")

Price columns converted to ₹
MRP range: ₹0.00 to ₹2600.00
Selling price range: ₹0.00 to ₹1399.00


## Step 2: Clean Text Columns

Product names may have trailing spaces.  
Category names must be consistent for grouping later.


In [12]:
# Strip whitespace from text columns
df['name'] = df['name'].str.strip()
df['Category'] = df['Category'].str.strip()

# Check categories
print("Categories found:", df['Category'].nunique())
print("\nCategory list:")
for cat in sorted(df['Category'].unique()):
    print(f"    - {cat}")

Categories found: 14

Category list:
    - Beverages
    - Biscuits
    - Chocolates & Candies
    - Cooking Essentials
    - Dairy, Bread & Batter
    - Fruits & Vegetables
    - Health & Hygiene
    - Home & Cleaning
    - Ice Cream & Desserts
    - Meats, Fish & Eggs
    - Munchies
    - Paan Corner
    - Packaged Food
    - Personal Care


## Step 3: Data Quality Checks

Before engineering new features, verify:

1. No MRP = 0 (invalid products)
2. No selling price > MRP (pricing error)
3. Discount percent in valid range (0–100)


In [13]:
# ── Duplicate check ─────────────────────────────────────────────
# Wrong approach: df.duplicated(['name', 'Category'])
# → removes legitimate different pack sizes (100g vs 500g)
#
# Correct approach: check ALL columns → only truly identical rows

# Step 1: Fully identical rows (all 9 columns same) — safe to drop
full_dupes = df.duplicated(keep=False).sum()
print(f"Fully identical rows: {full_dupes} → dropping these")
df = df.drop_duplicates().reset_index(drop=True)

# Step 2: Same name + category + weight = true duplicate SKU → drop
sku_dupes = df.duplicated(
    subset=['name', 'Category', 'weightInGms'], keep='first'
).sum()
print(f"Same name + category + weight: {sku_dupes} → dropping these")
df = df.drop_duplicates(
    subset=['name', 'Category', 'weightInGms']
).reset_index(drop=True)

# Step 3: Same name + category but different weight = different pack sizes → KEEP
multi_pack = df.groupby(['name', 'Category'])['weightInGms'].nunique()
print(f"\nProducts with multiple pack sizes (kept): {(multi_pack > 1).sum()}")
print(f"Final row count: {len(df)}")

Fully identical rows: 4 → dropping these
Same name + category + weight: 37 → dropping these

Products with multiple pack sizes (kept): 208
Final row count: 3693


In [14]:
# Check 1: MRP = 0 rows
zero_mrp = df[df['mrp'] == 0]
print(f"Rows with MRP = 0: {len(zero_mrp)}")
if len(zero_mrp) > 0:
    print(zero_mrp[['Category', 'name', 'mrp', 'discountedSellingPrice']].head())

# Check 2: Selling price > MRP (should not happen)
price_error = df[df['discountedSellingPrice'] > df['mrp']]
print(f"\nRows where selling price > MRP: {len(price_error)}")

# Check 3: Discount percent range
print(f"\nDiscount % range: {df['discountPercent'].min()} to {df['discountPercent'].max()}")
print(f"Items with 0% discount: {(df['discountPercent'] == 0).sum()}")

Rows with MRP = 0: 1
             Category                                       name  mrp  \
3572  Home & Cleaning  Cherry Blossom Liquid Shoe Polish Neutral 0.00   

      discountedSellingPrice  
3572                    0.00  

Rows where selling price > MRP: 0

Discount % range: 0 to 51
Items with 0% discount: 1162


Before engineering new features, verify:

1. No MRP = 0 (invalid products)
2. No selling price > MRP (pricing error)
3. Discount percent in valid range (0–100)


In [15]:
# Remove rows where MRP = 0 (no valid price = can't analyze)
rows_before = len(df)
df = df[df['mrp'] > 0].reset_index(drop=True)
rows_after = len(df)

print(f"Rows removed (MRP=0): {rows_before - rows_after}")
print(f"Final row count: {rows_after}")

Rows removed (MRP=0): 1
Final row count: 3692


## Step 4: Feature Engineering

Create 4 new columns needed for analysis:

| New Column        | Formula                                 | Purpose                       |
| ----------------- | --------------------------------------- | ----------------------------- |
| `discount_amount` | MRP - Selling Price                     | Absolute saving in ₹          |
| `price_per_100g`  | (MRP / weightInGms) × 100               | Compare value across products |
| `stock_status`    | Based on outOfStock + availableQuantity | Readable label                |
| `discount_tier`   | Bucket discount% into Low/Mid/High      | For grouping                  |


In [16]:
# How many rupees saved per item
df['discount_amount'] = df['mrp'] - df['discountedSellingPrice']

print("Discount amount column created.")
print(f"Avg discount: ₹{df['discount_amount'].mean():.2f}")
print(f"Max discount: ₹{df['discount_amount'].max():.2f}")
print(f"Items with no discount: {(df['discount_amount'] == 0).sum()}")

Discount amount column created.
Avg discount: ₹14.96
Max discount: ₹1201.00
Items with no discount: 1152


In [17]:
# Price per 100 grams — lets you compare value across different pack sizes
# Avoid divide by zero: only calculate where weightInGms > 0
df['price_per_100g'] = np.where(
    df['weightInGms'] > 0,
    (df['mrp'] / df['weightInGms']) * 100,
    np.nan
)

print("price_per_100g column created.")
print(f"Rows with valid price_per_100g: {df['price_per_100g'].notna().sum()}")
print(f"Avg price per 100g: ₹{df['price_per_100g'].mean():.2f}")

price_per_100g column created.
Rows with valid price_per_100g: 3688
Avg price per 100g: ₹101.93


In [18]:
# Cap extreme price_per_100g outliers (weight data errors)
p99 = df['price_per_100g'].quantile(0.99)
outlier_count = (df['price_per_100g'] > p99).sum()
df.loc[df['price_per_100g'] > p99, 'price_per_100g'] = np.nan

print(f"price_per_100g outliers capped to NaN: {outlier_count}")
print(f"New max: ₹{df['price_per_100g'].max():.2f}")

price_per_100g outliers capped to NaN: 36
New max: ₹1125.00


In [19]:
# Stock status — human readable label
def get_stock_status(row):
    if row['outOfStock']:
        return 'Out of Stock'
    elif row['availableQuantity'] <= 2:
        return 'Low Stock'
    else:
        return 'In Stock'

df['stock_status'] = df.apply(get_stock_status, axis=1)

# Discount tier — bucket the discount %
def get_discount_tier(pct):
    if pct == 0:
        return 'No Discount'
    elif pct <= 10:
        return 'Low (1-10%)'
    elif pct <= 20:
        return 'Mid (11-20%)'
    else:
        return 'High (21%+)'

df['discount_tier'] = df['discountPercent'].apply(get_discount_tier)

print("New columns added:")
print("\nstock_status:\n", df['stock_status'].value_counts())
print("\ndiscount_tier:\n", df['discount_tier'].value_counts())

New columns added:

stock_status:
 stock_status
In Stock        2709
Low Stock        533
Out of Stock     450
Name: count, dtype: int64

discount_tier:
 discount_tier
Low (1-10%)     1727
No Discount     1161
Mid (11-20%)     572
High (21%+)      232
Name: count, dtype: int64


In [20]:
# Rename columns → consistent snake_case, clear meaning
df = df.rename(columns={
    'Category': 'category',
    'name': 'product_name',
    'mrp': 'mrp_inr',
    'discountPercent': 'discount_pct',
    'availableQuantity': 'available_qty',
    'discountedSellingPrice': 'selling_price_inr',
    'weightInGms': 'weight_gms',
    'outOfStock': 'out_of_stock',
    'quantity': 'unit_qty'
})

print("Columns renamed.")
print("Final columns:", list(df.columns))

Columns renamed.
Final columns: ['category', 'product_name', 'mrp_inr', 'discount_pct', 'available_qty', 'selling_price_inr', 'weight_gms', 'out_of_stock', 'unit_qty', 'discount_amount', 'price_per_100g', 'stock_status', 'discount_tier']


In [21]:
# 4 products have weight_gms = 0 — NaN is intentional
print("Products with missing price_per_100g:")
print(df[df['price_per_100g'].isna()][['product_name', 'category', 'weight_gms']])

Products with missing price_per_100g:
                                           product_name              category  \
246                                        Keya Oregano    Cooking Essentials   
601                               Everest Saffron Kesar    Cooking Essentials   
755                                        Keya Oregano              Munchies   
1110                              Everest Saffron Kesar              Munchies   
1595                  Prasuma Momos -  Cheesy Spicy Veg         Packaged Food   
1981                  Prasuma Momos -  Cheesy Spicy Veg  Ice Cream & Desserts   
2367                  Prasuma Momos -  Cheesy Spicy Veg  Chocolates & Candies   
2792  Nivea Lip Balm Original Care for 24h Moisture ...         Personal Care   
2821           Maybelline New York Colossal Kajal Black         Personal Care   
2830  Stayfree Dry Max All Night XL Dry Cover Sanita...         Personal Care   
2836                 Pampers Premium Care Pants - Large         Persona

In [22]:
print("=== FINAL DATASET SUMMARY ===")
print(f"Shape: {df.shape}")
print(f"\nColumn types:\n{df.dtypes}")
print(f"\nNull counts:\n{df.isnull().sum()}")

=== FINAL DATASET SUMMARY ===
Shape: (3692, 13)

Column types:
category              object
product_name          object
mrp_inr              float64
discount_pct           int64
available_qty          int64
selling_price_inr    float64
weight_gms             int64
out_of_stock            bool
unit_qty               int64
discount_amount      float64
price_per_100g       float64
stock_status          object
discount_tier         object
dtype: object

Null counts:
category              0
product_name          0
mrp_inr               0
discount_pct          0
available_qty         0
selling_price_inr     0
weight_gms            0
out_of_stock          0
unit_qty              0
discount_amount       0
price_per_100g       40
stock_status          0
discount_tier         0
dtype: int64


In [23]:
# Summary stats for cleaned numeric columns
summary = df[['mrp_inr', 'selling_price_inr', 'discount_pct', 
              'discount_amount', 'price_per_100g']].describe().round(2)
print(summary)

       mrp_inr  selling_price_inr  discount_pct  discount_amount  \
count  3692.00            3692.00       3692.00          3692.00   
mean    156.71             141.75          7.63            14.96   
std     160.18             137.40          9.23            40.87   
min      10.00               9.00          0.00             0.00   
25%      60.00              55.00          0.00             0.00   
50%     110.00             103.50          6.00             5.00   
75%     200.00             185.00         10.00            16.00   
max    2600.00            1399.00         51.00          1201.00   

       price_per_100g  
count         3652.00  
mean            79.79  
std            110.38  
min              1.72  
25%             25.00  
50%             49.50  
75%             89.00  
max           1125.00  


In [24]:
# Create output folder if not exists
os.makedirs('../data', exist_ok=True)

# Save clean file
df.to_csv('../data/zepto_clean.csv', index=False)
print(f"Clean data saved: data/zepto_clean.csv")
print(f"Final rows: {len(df)}")
print(f"Final columns: {len(df.columns)}")

Clean data saved: data/zepto_clean.csv
Final rows: 3692
Final columns: 13


## Phase 1 Complete ✓

Clean dataset exported: `zepto_clean.csv`

- Rows: 3,731 (1 removed — MRP = 0)
- Columns: 13 (9 original + 4 engineered)
- Next → Phase 2 EDA: `02_zepto_eda.ipynb`


## Key Findings from Wrangling

| Metric | Value | Insight |
|---|---|---|
| Out of stock SKUs | 453 (12.1%) | 1 in 8 products unavailable |
| Zero discount products | 1,177 (31%) | Majority products at full MRP |
| Avg discount saved | ₹14.88 | Low savings per product |
| Max discount saved | ₹1,201 | Extreme outlier — premium product |
| Low stock SKUs | 545 (14.6%) | Risk of stockout soon |
| Avg MRP | ₹156.84 | Mid-range grocery pricing |
| Products with no weight data | 4 | Cannot compute price/100g |